# Agent_NanoscribeV1 - Pipeline Demo

This notebook demonstrates the complete pipeline from natural language prompt to GWL output.

## Pipeline Overview
1. **Geometry Agent**: Prompt -> Unit Cell JSON
2. **Endpoint Generator**: Unit Cell -> Scan Endpoints
3. **GWL Serializer**: Endpoints -> GWL Files
4. **Render Generator**: Visualization
5. **Redesign Agent**: Iterative Modifications

## Setup

In [1]:
import os
import sys
import json
import traceback
from pathlib import Path
from datetime import datetime
from langfuse import Langfuse, propagate_attributes
from langfuse.langchain import CallbackHandler

# Add current directory to path
NOTEBOOK_DIR = Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))

print(f"Working directory: {NOTEBOOK_DIR}")

Working directory: /Users/justin/Notebooks/Inspire/Inspire/Agent_NanoscribeV1


# Before you proceed:
Create a file "Docs/env.json" based on the following template to hold API keys and model specifications:  

In [2]:
from geometry_agent import find_env_file
ENV_DICT = find_env_file(Path("./"))
os.environ.update(ENV_DICT)

[CONFIG] Found API key at: /Users/justin/Notebooks/Inspire/Inspire/Agent_NanoscribeV1/Docs/env.json
[CONFIG] Found API key at: Docs/env.json


## Step 0: Authenticate to LangFuse

In [ ]:
langfuse_client = Langfuse()
try:
  langfuse_client.auth_check()
  print("Langfuse client is authenticated and ready!")
except Exception as e:
  print("Authentication failed to LangFuse. Please check your credentials and host. Disabling LangFuse.")
  print(traceback.format_exc())
  langfuse_client = None

## Step 1: Define Your Prompt

Describe the geometry you want to create in natural language.

In [4]:
# Define your geometry prompt
USER_PROMPT = """
Create a 5x5 array of solid cylindrical posts. 
Each cylinder is 10 um tall and 4 um in diameter. 
The cylinders are arranged in a square grid with 15 um center-to-center spacing.
"""

print("Prompt:")
print(USER_PROMPT)

Prompt:

Create a 5x5 array of solid cylindrical posts. 
Each cylinder is 10 um tall and 4 um in diameter. 
The cylinders are arranged in a square grid with 15 um center-to-center spacing.



## Step 2: Run Geometry Agent

The LLM analyzes the prompt and generates structured unit cell JSON.

In [ ]:
from geometry_agent import build_graph, AgentState

# Build the LangGraph workflow
graph = build_graph()

# Create initial state
initial_state = AgentState(
    prompt=USER_PROMPT.strip(),
    category="",
    result={},
    output_path="",
    token_usage={}
)

langfuse_handler = CallbackHandler()

with propagate_attributes(tags = [initial_state["category"]]):
  
  # Run the graph
  print("Running Geometry Agent...")
  session_id = datetime.now().strftime("jupyter_run_%Y-%m-%d_%H:%M:%S")
  final_state = graph.invoke(initial_state, config={
    "callbacks": [langfuse_handler],
    "metadata": {
        "langfuse_user_id": "test_user",
        "langfuse_session_id": session_id,
        "langfuse_tags": [initial_state["category"]]
    }
  })

print(f"\nJob Name: {final_state['result']['job_name']}")
print(f"Tokens Used: {final_state['token_usage']['total_tokens']}")
print(f"Output saved to: {final_state['output_path']}")

## Step 3: Examine Unit Cell JSON

Inspect the generated geometry structure.

In [6]:
unit_cell_data = final_state['result']

print("=" * 60)
print("UNIT CELL JSON")
print("=" * 60)

print(f"\nJob Name: {unit_cell_data['job_name']}")
print(f"\nUnit Cell Location: {unit_cell_data['unit_cell']['location']}")

print(f"\nComponents ({len(unit_cell_data['unit_cell']['components'])}):")  
for i, comp in enumerate(unit_cell_data['unit_cell']['components']):
    print(f"  [{i}] {comp['type']}")
    print(f"      Center: {comp['center']}")
    print(f"      Dimensions: {comp['dimensions']}")

print(f"\nGlobal Info:")
print(f"  Pattern Type: {unit_cell_data['global_info']['pattern_type']}")
print(f"  Repetitions: {unit_cell_data['global_info']['repetitions']}")
print(f"  Spacing: {unit_cell_data['global_info']['spacing']}")

UNIT CELL JSON

Job Name: 5x5 Solid Cylindrical Posts

Unit Cell Location: bottom_centered

Components (1):
  [0] cylinder
      Center: [0, 0, 0]
      Dimensions: {'height_um': 10, 'diameter_um': 4}

Global Info:
  Pattern Type: array
  Repetitions: {'x': 5, 'y': 5, 'z': 1}
  Spacing: {'x_um': 15, 'y_um': 15, 'z_um': 0, 'pattern_description': 'arranged in a square grid with 15 um center-to-center spacing'}


## Step 4: Check Print Parameters

In [ ]:
from endpoint_generator import load_print_parameters

param_file = NOTEBOOK_DIR / "PrintParameters.txt"
print_params = load_print_parameters(param_file)

print("=" * 60)
print("PRINT PARAMETERS")
print("=" * 60)
for key, value in print_params.items():
    print(f"  {key}: {value}")

## Step 5: Generate Endpoints

Convert unit cell to scan path endpoints.

In [ ]:
from endpoint_generator import generate_endpoint_json

print("Generating endpoints...")
endpoint_data = generate_endpoint_json(unit_cell_data, print_params)

print("\n" + "=" * 60)
print("ENDPOINT SUMMARY")
print("=" * 60)
print(f"Job: {endpoint_data['job_name']}")
print(f"Total Layers: {len(endpoint_data['layers'])}")

total_segments = sum(len(layer['segments']) for layer in endpoint_data['layers'])
print(f"Total Segments: {total_segments}")

if endpoint_data['layers']:
    z_min = endpoint_data['layers'][0]['z_um']
    z_max = endpoint_data['layers'][-1]['z_um']
    print(f"Z Range: {z_min:.2f} to {z_max:.2f} um")

## Step 6: Generate GWL Files

In [ ]:
from gwl_serializer import generate_gwl_files, generate_master_gwl, load_gwl_parameters

# Create output directory
output_dir = Path(final_state['output_path']).parent
gwl_dir = output_dir / "GWL"
gwl_dir.mkdir(exist_ok=True)

gwl_params = load_gwl_parameters(param_file)

print("=" * 60)
print("GWL PARAMETERS")
print("=" * 60)
for key, value in gwl_params.items():
    print(f"  {key}: {value}")

print("\nGenerating GWL files...")
gwl_files = generate_gwl_files(endpoint_data, gwl_params, gwl_dir)

master_path = gwl_dir / f"{endpoint_data['job_name']}_master.gwl"
generate_master_gwl(gwl_files, gwl_params, master_path)

print(f"\nGenerated {len(gwl_files)} layer files")
print(f"Master file: {master_path.name}")

## Step 7: Generate Renders

In [ ]:
from render_generator import generate_all_renders

# Save endpoints for render generator
endpoint_file = output_dir / "output.json"
with open(endpoint_file, 'w') as f:
    json.dump(endpoint_data, f, indent=2)

render_dir = output_dir / "Renders"

print("Generating renders...")
render_result = generate_all_renders(endpoint_file, param_file, render_dir)

print(f"\nGenerated renders in: {render_dir}")

## Step 8: Display Renders

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

render_files = sorted(render_dir.glob("*.png"))

# Display unit cell renders
unit_cell_renders = [f for f in render_files if "unit_cell" in f.name]

if unit_cell_renders:
    fig, axes = plt.subplots(1, len(unit_cell_renders), figsize=(18, 6))
    if len(unit_cell_renders) == 1:
        axes = [axes]
    
    for ax, render_path in zip(axes, unit_cell_renders):
        img = mpimg.imread(render_path)
        ax.imshow(img)
        ax.set_title(render_path.stem.replace("_", " ").title())
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

---
## Redesign Demo

Now let's demonstrate the redesign system.

In [ ]:
# Add redesign package to path
sys.path.insert(0, str(NOTEBOOK_DIR / "redesign"))

from redesign_agent import RedesignSession

# Create redesign session
redesign_output = NOTEBOOK_DIR / "outputs" / "redesigns"
session = RedesignSession("demo", redesign_output)

# Initialize from the unit cell we just created
unit_cell_file = output_dir / "unit_cell.json"
session.initialize_from_unit_cell(unit_cell_file, param_file, "Initial cylinder array")

print("\nRedesign session initialized!")
print(f"Output directory: {session.output_dir}")

In [ ]:
# Apply a redesign
REDESIGN_PROMPT = "Make the cylinders 50% taller"

print(f"Redesign Prompt: {REDESIGN_PROMPT}")
print("\nApplying redesign...")

new_variant = session.apply_redesign(REDESIGN_PROMPT)

In [ ]:
# Compare original and redesigned
original = session.design_variants[0]
redesigned = session.design_variants[1]

print("=" * 60)
print("COMPARISON: Original vs Redesigned")
print("=" * 60)

orig_comp = original['unit_cell_json']['unit_cell']['components'][0]
new_comp = redesigned['unit_cell_json']['unit_cell']['components'][0]

print(f"\nOriginal cylinder:")
print(f"  Height: {orig_comp['dimensions'].get('height_um', 'N/A')} um")
print(f"  Diameter: {orig_comp['dimensions'].get('diameter_um', 'N/A')} um")

print(f"\nRedesigned cylinder:")
print(f"  Height: {new_comp['dimensions'].get('height_um', 'N/A')} um")
print(f"  Diameter: {new_comp['dimensions'].get('diameter_um', 'N/A')} um")

In [ ]:
# Print variant history
session.print_history()

## Summary

This notebook demonstrated the complete Agent_NanoscribeV1 pipeline:

1. **Geometry Agent**: Converted natural language to structured JSON
2. **Endpoint Generator**: Created layer-by-layer scan paths
3. **GWL Serializer**: Generated Nanoscribe-compatible files
4. **Render Generator**: Created visualization images
5. **Redesign Agent**: Applied iterative modifications

The generated GWL files are ready to load into NanoWrite for fabrication.